# 🧬 ECABSD — Full Training Pipeline (Auto-Save Edition)

> **Checkpoints auto-save to `/kaggle/working/` every epoch so they survive session restarts.**

### Pipeline:
1. ✅ GPU check & install dependencies
2. ✅ Clone ECABSD from GitHub
3. ✅ Download DB5 benchmark structures
4. ✅ Prepare dataset (PDB → residue graphs with ESM-2)
5. ✅ Train V3 model — checkpoint auto-saves to top-level `/kaggle/working/`
6. ✅ Scientific validation suite
7. ✅ Export all results — checkpoint + figures + JSON at top-level

**⚠️ Click `Save Version` as soon as you see epochs running in Cell 5!**

## ⚙️ Cell 1 — GPU Check & Install Dependencies

In [ ]:
import subprocess, sys, os

# Verify GPU
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else '❌ No GPU — enable in Notebook Settings!')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

def run(cmd):
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if r.stdout: print(r.stdout[-1000:])
    if r.returncode != 0 and r.stderr: print('ERR:', r.stderr[-500:])
    return r.returncode

print('\nInstalling dependencies...')
run('pip install -q torch-geometric')
run('pip install -q biopython pydssp transformers==4.40.2 '
    'fastapi uvicorn typer pyyaml scikit-learn tqdm '
    'matplotlib seaborn python-multipart pandas scipy')
print('\n✅ All dependencies installed!')

## 📁 Cell 2 — Clone ECABSD Repository

In [ ]:
import os, sys, subprocess

REPO_URL = 'https://github.com/amanigreeva/ECABSD.git'
WORK_DIR = '/kaggle/working/ecabsd'   # flat path — no nested folders!

# Top-level output dir — files here survive Save Version
TOP      = '/kaggle/working'
CKPT_DIR = TOP                        # ← checkpoints go HERE
LOGS_DIR = os.path.join(TOP, 'logs')
RES_DIR  = os.path.join(TOP, 'results')

for d in [CKPT_DIR, LOGS_DIR, RES_DIR]:
    os.makedirs(d, exist_ok=True)

if os.path.exists(WORK_DIR):
    r = subprocess.run(f'git -C {WORK_DIR} pull origin main',
                       shell=True, capture_output=True, text=True)
    print(r.stdout)
    print('✅ Pulled latest from GitHub')
else:
    r = subprocess.run(f'git clone {REPO_URL} {WORK_DIR}',
                       shell=True, capture_output=True, text=True)
    if r.returncode == 0:
        print('✅ Cloned ECABSD successfully')
    else:
        print('ERROR:', r.stderr)
        raise RuntimeError('Clone failed')

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)

subprocess.run('ls -la', shell=True)
print(f'\nWorking dir: {os.getcwd()}')
print(f'Checkpoints will save to: {CKPT_DIR}')

## 📥 Cell 3 — Download DB5 & Prepare Dataset
> Takes ~40–60 min (ESM-2 embeddings for all PDBs)

In [ ]:
import os, subprocess

RAW_DIR  = '/kaggle/working/raw_pdbs'
PROC_DIR = '/kaggle/working/processed'
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROC_DIR, exist_ok=True)

# ── Download DB5 structures ─────────────────────────────────────────────────
existing = [f for f in os.listdir(RAW_DIR) if f.endswith('.pdb')]
if len(existing) < 10:
    print('Downloading DB5 benchmark structures...')
    r = subprocess.run(
        f'python scripts/download_benchmarks.py --output-dir {RAW_DIR}',
        shell=True, capture_output=True, text=True)
    print(r.stdout[-2000:])
    if r.returncode != 0:
        print('WARN:', r.stderr[-500:])
else:
    print(f'✅ {len(existing)} PDB files already present')

existing = [f for f in os.listdir(RAW_DIR) if f.endswith('.pdb')]
print(f'Total PDB files: {len(existing)}')

# ── Build residue graphs ─────────────────────────────────────────────────────
graphs = [f for f in os.listdir(PROC_DIR) if f.endswith('.pt')]
if len(graphs) < 50:
    print('\nBuilding residue graphs with ESM-2 embeddings...')
    print('(This takes ~40-60 min on GPU T4)\n')
    subprocess.run(
        f'python scripts/prepare_dataset.py '
        f'--pdb-dir {RAW_DIR} '
        f'--output-dir {PROC_DIR} '
        f'--cutoff 4.5',
        shell=True)
else:
    print(f'✅ {len(graphs)} graph files already prepared')

graphs = [f for f in os.listdir(PROC_DIR) if f.endswith('.pt')]
print(f'\n✅ Graph files ready: {len(graphs)}')

import pandas as pd
SPLITS_CSV = os.path.join(WORK_DIR, 'data', 'splits.csv')
if os.path.exists(SPLITS_CSV):
    df = pd.read_csv(SPLITS_CSV)
    print('\nSplit counts:')
    print(df['split'].value_counts())

## ⚙️ Cell 4 — Configure Paths (saves to top-level)

In [ ]:
import yaml, os

SPLITS_CSV = os.path.join(WORK_DIR, 'data', 'splits.csv')

with open('config.yaml') as f:
    cfg = yaml.safe_load(f)

# ── Override ALL paths to top-level /kaggle/working/ ───────────────────────
cfg['data']['processed_dir']    = '/kaggle/working/processed'
cfg['data']['splits_csv']       = SPLITS_CSV
cfg['paths']['checkpoints_dir'] = '/kaggle/working'   # TOP-LEVEL!
cfg['paths']['logs_dir']        = '/kaggle/working/logs'
cfg['paths']['results_dir']     = '/kaggle/working/results'

# ── Training params ─────────────────────────────────────────────────────────
cfg['training']['epochs']                = 100
cfg['training']['num_workers']           = 2
cfg['training']['early_stopping_patience'] = 30

with open('config_kaggle.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False, sort_keys=False)

print('✅ Kaggle config written!')
print(f"  processed_dir    : {cfg['data']['processed_dir']}")
print(f"  splits_csv       : {cfg['data']['splits_csv']}")
print(f"  checkpoints_dir  : {cfg['paths']['checkpoints_dir']}")
print(f"  epochs           : {cfg['training']['epochs']}")
print(f"\n✅ Checkpoint will save to: /kaggle/working/best_model_v3.pt")

## 🚀 Cell 5 — Train V3 Model (Auto-Save every epoch)

> **⚠️ As soon as you see epoch logs below, click `Save Version` button (top right)!**  
> This commits the notebook output so your checkpoint is permanently saved.

In [ ]:
import sys, os, shutil, torch
sys.path.insert(0, WORK_DIR)

# ── Auto-backup hook: copies best_model_v3.pt to top-level after every save ─
# The training loop already saves to /kaggle/working/ directly via config.
# This watchdog also saves periodic epoch backups every 10 epochs.
import threading, time

CKPT_SRC = '/kaggle/working/best_model_v3.pt'
_stop_watchdog = threading.Event()

def _watchdog():
    """Background thread: every 5 min, print checkpoint status."""
    while not _stop_watchdog.is_set():
        time.sleep(300)  # every 5 minutes
        if os.path.exists(CKPT_SRC):
            size = os.path.getsize(CKPT_SRC) / 1e6
            mtime = time.strftime('%H:%M:%S',
                time.localtime(os.path.getmtime(CKPT_SRC)))
            print(f'[AutoSave] ✅ best_model_v3.pt ({size:.1f} MB) '
                  f'last updated at {mtime}')

t = threading.Thread(target=_watchdog, daemon=True)
t.start()

# ── Run training ─────────────────────────────────────────────────────────────
from train import run_training

print('=' * 60)
print('  ECABSD V3 — Training Started')
print('  Checkpoint: /kaggle/working/best_model_v3.pt')
print('  ⚠️  Click SAVE VERSION now to lock in your outputs!')
print('=' * 60)

run_training(config_path='config_kaggle.yaml')

_stop_watchdog.set()

# ── Final verification ───────────────────────────────────────────────────────
print('\n' + '=' * 60)
if os.path.exists(CKPT_SRC):
    size = os.path.getsize(CKPT_SRC) / 1e6
    ckpt = torch.load(CKPT_SRC, map_location='cpu')
    print(f'✅ CHECKPOINT SAVED: best_model_v3.pt ({size:.1f} MB)')
    print(f"   Best Val F1   : {ckpt.get('best_val_f1', 'N/A'):.4f}")
    print(f"   Best Threshold: {ckpt.get('best_threshold', 'N/A'):.4f}")
    print(f"   Epoch         : {ckpt.get('epoch', 'N/A')}")
else:
    print('⚠️ Checkpoint not found at top level — searching...')
    import glob
    found = glob.glob('/kaggle/**/*.pt', recursive=True)
    for f in found:
        print(f'  Found: {f} ({os.path.getsize(f)/1e6:.1f} MB)')
        shutil.copy2(f, CKPT_SRC)
        print(f'  ✅ Copied to /kaggle/working/best_model_v3.pt')
print('=' * 60)

## 🔬 Cell 6 — Scientific Validation Suite

In [ ]:
import subprocess, os

CKPT = '/kaggle/working/best_model_v3.pt'

if not os.path.exists(CKPT):
    print('❌ No checkpoint found. Run Cell 5 first.')
else:
    scripts = [
        f'python scripts/scientific_validation.py --checkpoint {CKPT} --output-dir /kaggle/working/results',
        f'python scripts/error_analysis.py       --checkpoint {CKPT} --output-dir /kaggle/working/results',
        f'python scripts/hotspot_validation.py   --checkpoint {CKPT} --output-dir /kaggle/working/results',
        f'python scripts/calibration_analysis.py --checkpoint {CKPT} --output-dir /kaggle/working/results',
    ]
    for cmd in scripts:
        name = cmd.split()[1].split('/')[-1]
        print(f'\n[Running] {name}...')
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
        print(r.stdout[-1500:])
        if r.returncode != 0:
            print('WARN:', r.stderr[-500:])

    print('\n✅ Scientific validation complete!')
    print('Results saved to /kaggle/working/results/')

## 📦 Cell 7 — Export ALL Outputs to Top-Level
> Run this to make everything visible and downloadable in the Output panel

In [ ]:
import os, glob, shutil, zipfile

TOP = '/kaggle/working'

def find(name):
    matches = glob.glob(f'/kaggle/working/**/{name}', recursive=True)
    # exclude if already at top level
    matches = [m for m in matches
               if os.path.dirname(m) != TOP]
    return matches[0] if matches else None

# ── Copy all key files to /kaggle/working/ top-level ────────────────────────
targets = [
    'best_model_v3.pt',
    'epoch_20.pt',
    'epoch_40.pt',
    'epoch_60.pt',
    'epoch_80.pt',
    'epoch_100.pt',
    'training_history_v3.json',
    'statistical_validation.json',
    'calibration_stats.json',
    'hotspot_correlations.json',
    'error_analysis_report.json',
]

print('📦 Exporting files to /kaggle/working/ (top-level):')
for name in targets:
    dst = os.path.join(TOP, name)
    if os.path.exists(dst):
        size = os.path.getsize(dst) / 1e6
        print(f'  ✅ {name} ({size:.1f} MB) — already at top level')
    else:
        src = find(name)
        if src:
            shutil.copy2(src, dst)
            size = os.path.getsize(dst) / 1e6
            print(f'  ✅ {name} ({size:.1f} MB) — copied from {src}')
        else:
            print(f'  ⚠️  {name} — not found')

# ── Zip all figures ──────────────────────────────────────────────────────────
figures = glob.glob('/kaggle/working/**/figures/*.png', recursive=True)
if figures:
    zip_path = os.path.join(TOP, 'ecabsd_figures.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for fig in figures:
            zf.write(fig, os.path.basename(fig))
    print(f'  ✅ ecabsd_figures.zip ({len(figures)} figures)')

# ── Zip results folder ───────────────────────────────────────────────────────
results_zip = os.path.join(TOP, 'ecabsd_results.zip')
with zipfile.ZipFile(results_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(os.path.join(TOP, 'results')):
        for f in files:
            fp = os.path.join(root, f)
            zf.write(fp, os.path.relpath(fp, TOP))
print(f'  ✅ ecabsd_results.zip')

# ── Final summary ────────────────────────────────────────────────────────────
print('\n📁 Files now visible in Output panel:')
for f in sorted(os.listdir(TOP)):
    fp = os.path.join(TOP, f)
    if os.path.isfile(fp):
        print(f'   📄 {f}  ({os.path.getsize(fp)/1e6:.1f} MB)')

print('\n✅ Done! Refresh the Output panel on the right to download!')